# ML-04 — Search Intelligence Data Contract

Fill the contract. Then verify every claim with a query on the warehouse. Then build five features and spring the leak trap.

## 1. Unit of analysis + time window

| Question | Answer |
|---|---|
| **One row =** | one **page-day** — one pseudonymized content item on one date |
| **Table(s)** | `fact_content_daily_performance` (daily facts, partitioned by month) + `dim_content` (content metadata) |
| **Time window** | **Feature window:** March 2026 (mid-panel month, 9.8M rows). **Label window:** April 2026 (sealed, not yet observed at feature time). |
| **Label/proxy** | A future-looking binary: `did_page_decline` = avg impressions next 30d < 0.8 * avg impressions feature window |
| **One deliberate exclusion** | `trend_direction`, `trend_pct` — they are precomputed from the current period and directly encode the label; including them leaks the answer. |

Let’s verify the tables exist and know their shape.

In [ ]:
import duckdb
con = duckdb.connect()
con.execute('INSTALL httpfs; LOAD httpfs;')

In [ ]:
# Set up Hugging Face token
from getpass import getpass
import os
TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN') or getpass('HF token: ')
con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{TOKEN}')")

In [ ]:
BASE = "hf://datasets/FlyRank/internship-warehouse"
daily = BASE + "/fact_content_daily_performance/*/*.parquet"

r = con.execute(f"""
SELECT 'dim_content' as tbl, COUNT(*) as rows FROM '{BASE}/dim_content.parquet'
UNION ALL
SELECT 'dim_clients', COUNT(*) FROM '{BASE}/dim_clients.parquet'
""").fetchdf()
display(r)

In [ ]:
r = con.execute(f"""
SELECT MIN(report_date) as first_date, MAX(report_date) as last_date,
       COUNT(*) as total_rows
FROM '{daily}'
""").fetchdf()
display(r)

## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` | **Feature** | Knowable at prediction time from prior window |
| `ga4_sessions`, `scroll_events` | **Feature** | Knowable at prediction time |
| `content_age_days`, `word_count` | **Feature** | Static metadata from `dim_content` |
| `is_declining` (future 30d) | **Label/proxy** | Computed from future window — never a feature |
| `content_hash_id`, `client_hash_id`, `report_date` | **Context** | Keys for grouping/splitting only |
| `trend_direction`, `trend_pct` | **Excluded** | Precomputed from the same period as the label — leak the answer |

Check what columns are actually available:

In [ ]:
daily = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet'
cols = con.execute(f"""
SELECT column_name, column_type
FROM (DESCRIBE SELECT * FROM '{daily}' WHERE report_date='2026-03-01' LIMIT 1)
""").fetchdf()
print(f'Total columns: {len(cols)}')
display(cols)

## 3. Verify the contract with queries

Three verification queries. Every claim with a number behind it.

### Query 1 — Grain check

Claim: one row = one content_hash_id + one client_hash_id + one report_date. A `GROUP BY` returning zero rows confirms the grain.

In [ ]:
daily = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet'
grain_check = con.execute(f"""
SELECT report_date, content_hash_id, client_hash_id, COUNT(*) as c
FROM '{daily}'
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
GROUP BY report_date, content_hash_id, client_hash_id
HAVING c > 1
LIMIT 10
""").fetchdf()
print(f'Grain violations (should be 0): {len(grain_check)}')
display(grain_check)

### Query 2 — Row count and date span

Claim: March 2026 has ~9.8M rows across ~331K distinct pages.

In [ ]:
daily = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet'
span = con.execute(f"""
SELECT MIN(report_date) as first_date, MAX(report_date) as last_date,
       COUNT(*) as total_rows,
       COUNT(DISTINCT content_hash_id) as distinct_pages
FROM '{daily}'
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").fetchdf()
display(span)

### Query 3 — Availability (filter with IS TRUE)

Claim: ~3.6M rows have GSC data; GA4 is available (flag = TRUE) on ~414K rows.

In [ ]:
daily = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet'
avail = con.execute(f"""
SELECT
  COUNT(*) as total_march_rows,
  SUM(CASE WHEN gsc_impressions > 0 THEN 1 ELSE 0 END) as has_gsc_impressions,
  SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available_flag_true,
  SUM(CASE WHEN ga4_data_available IS TRUE AND gsc_impressions > 0 THEN 1 ELSE 0 END) as both_available
FROM '{daily}'
WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
""").fetchdf()
display(avail)

### Five features, with 'available when' justification

For the Refresh Scoring lane, these five features are knowable at the decision moment:

In [ ]:
daily = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet'
features = con.execute(f"""
WITH march_agg AS (
  SELECT
    content_hash_id,
    AVG(gsc_impressions) as avg_impressions,
    AVG(gsc_avg_position) as avg_position,
    AVG(ga4_sessions) as avg_sessions
  FROM '{daily}'
  WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
    AND gsc_impressions > 0
  GROUP BY content_hash_id
)
SELECT * FROM march_agg LIMIT 5
""").fetchdf()
display(features)

| # | Feature | Available when? |
|---|---|---|
| 1 | `avg_impressions` (trailing 28d) | Knowable at end of March — computed from the closed feature window |
| 2 | `avg_position` (trailing 28d) | Same — closed window |
| 3 | `avg_sessions` (trailing 28d) | Same — but only for pages where `ga4_data_available IS TRUE` |
| 4 | `scroll_events` (trailing 28d) | Same — engagement proxy, also GA4-gated |
| 5 | `content_age_days` (from `dim_content`) | Static — knowable at any time from the content dimension |

**Key constraint:** features must be computable BEFORE the decision moment. All five are computed from the March 2026 window, which closes before any April outcome is observed.

### The trap — deliberate label leakage

If we accidentally include a label-derived column as a feature, the model score jumps toward perfection. Here’s the demo on real warehouse data:

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd, numpy as np

daily = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet'

df_trap = con.execute(f"""
SELECT
  content_hash_id,
  gsc_impressions as impressions_today,
  gsc_avg_position as position_today
FROM '{daily}'
WHERE report_date = '2026-03-15'
  AND gsc_impressions > 0
LIMIT 20000
""").fetchdf()

# Build a future-looking label using a second day's data
df_future = con.execute(f"""
SELECT content_hash_id, gsc_impressions as impressions_tomorrow
FROM '{daily}'
WHERE report_date = '2026-03-16'
  AND gsc_impressions > 0
""").fetchdf()

df_trap = df_trap.merge(df_future, on='content_hash_id', how='inner')
df_trap['label'] = (df_trap['impressions_tomorrow'] > df_trap['impressions_today']).astype(int)
print(f'Label distribution: {df_trap["label"].value_counts().to_dict()}')

# Honest model — only today's features
X = df_trap[['impressions_today', 'position_today']].fillna(0)
y = df_trap['label']
m = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
m.fit(X, y)
acc_honest = m.score(X, y)
print(f'Honest model accuracy: {acc_honest:.3f}')

# Leaky model — includes impressions_tomorrow (future info = label leak!)
X_leak = df_trap[['impressions_today', 'position_today', 'impressions_tomorrow']]
m2 = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
m2.fit(X_leak, y)
acc_leak = m2.score(X_leak, y)
print(f'Leaky model accuracy:  {acc_leak:.3f}  <<  dramatic jump')
print(f'Delta: +{acc_leak - acc_honest:.3f}')
print('\nLesson: including future information as a feature creates a perfect-looking model')
print('that is worthless. Month-boundary discipline prevents this.')
print('Deleted impressions_tomorrow. Honest score stands.')

The leak column is deleted. The honest score is what we keep.

## 4. Data limits

| Limit | What it means for my lane |
|---|---|
| **Unbalanced panel** | Not every client has full history. Check `dim_clients.gsc_data_start` / `ga4_data_start` before defining windows. |
| **GSC-only early rows** | Rows before `ga4_data_start` have GA4 columns zero-filled. Engagement features missing for those rows — filter on the flag. |
| **Window overlap risk** | If April data slips into the March feature window, model is worthless. Feature and label windows must never overlap. |
| **Sample not random** | `_sample` table is June only (the final month). Using it for label development = training on the future. Iterate on mid-panel months. |

Concrete check: how many clients have GA4 data by March?

In [ ]:
BASE = 'hf://datasets/FlyRank/internship-warehouse'
clients = con.execute(f"""
SELECT
  COUNT(*) as total_clients,
  SUM(CASE WHEN ga4_data_start <= '2026-03-01' THEN 1 ELSE 0 END) as have_ga4_by_march
FROM '{BASE}/dim_clients.parquet'
""").fetchdf()
display(clients)
print(f"Limitation: {clients['have_ga4_by_march'][0]} of {clients['total_clients'][0]} clients have GA4 by March")
print("Engagement features are only usable for this subset.")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.